# T2 step 7 · Every number in the reports, checked against the damage facts and the guidelines (your PC, CPU)

Roadmap T2 step 7, built 20 Nov 2026, at Abrar's go. It reads T2 step 5's saved runs; it runs no model.

**Why:** T2 step 5 gave the reports the system's own damage facts (C4's counts, hot spots and safe areas), and both
scenes passed the five pass rules. But the 7B wrote numbers the facts don't have: in all six Santa Rosa reports
"96% of the area" for 96 of 144 destroyed patches (67%), in one Turkey report "283,81 pixels" for 28,381, and staff
numbers no source gives ([step 5's results](results/t2_step5_c4/README.md)). The one-chain goal asks for zero
contradictions (roadmap §1b), so every number must be checked, not just the ones a reader notices.

**How:** for each report, its sources are rebuilt from the run's own logs, exactly as the 7B saw them:
- the damage facts and the code's map reading, from `run_log.txt` (the flow prints both);
- the guideline passages, from the section IDs each search returned (`llm_calls.jsonl`), printed as the search
  tool printed them (with the document and the pages);
- not the photo descriptions or the earlier reports, which the 7B wrote.

Every number in every model-written section is then read with what it measures and checked by
[number_check.py](disasteller_offline/number_check.py), the same rules fix (b) uses during writing:
- a position, a pixel figure, a percentage or a count of buildings or patches must match the damage facts; a
  percentage may be a share of the counts (within 1 point);
- any other number must be in the facts or in the passages shown to that report.

The code-written grade section is skipped. **Wrong** means a damage fact stated with the wrong value (a
contradiction); **unsourced** means a number in neither the facts nor the passages.

**The rule (roadmap T2 step 7, fixed 26 Oct):** a run passes only with zero wrong numbers. Unsourced numbers are
counted beside it; fix (b) also asks again for them.

**Expectation for step 5's first run (fixed before this notebook ran, from the first look by eye):** it fails.
At least 7 wrong numbers (the six "96%" and "283,81"), and several unsourced staff numbers in the two
human-resource reports. *(Result: it fails, with 13 wrong, since "96%" is in 12 sections, and 5 unsourced, all
in Santa Rosa's human-resource report.)*

**Run 2 of step 5** ([t2_step5_c4_fix_run.ipynb](t2_step5_c4_fix_run.ipynb), fixes (a) and (b), 20 Nov) is
checked the same way. **Its expectation, fixed before it ran:** at most 2 wrong and at most 2 unsourced numbers
(target 0), and no "96%". *(Result: 0 and 0 of 191, so it passes. Read by eye, though, it reuses damage counts
as staff numbers: "54 medical personnel" for 54 intact buildings.)*

**Rule version 2** (20 Nov, evening, Abrar's go; fixed before run 3): a damage count used for staff, teams,
shelters or other resources is **misused**, and counts as wrong. Every run is measured under both versions, side
by side ([number_check.py](disasteller_offline/number_check.py) has the exact rule).
- **Expectation for rule 2 on runs 1 and 2, fixed before measuring:** run 1 still fails (13 wrong, about 0
  misused); run 2 fails, with about 7 misused.
- **Run 3** ([t2_step5_c4_fix2_run.ipynb](t2_step5_c4_fix2_run.ipynb)) has rule 2 in the check while writing.
  **Its expectation, fixed before it ran:** at most 2 wrong, unsourced or misused numbers in all (target 0).
  *(Result: 10, all in Turkey's staffing sections, so the expectation is missed: asked again, the 7B kept the same
  numbers, because those sections ask for staff and shelter numbers the facts don't give.)*
- **Run 4** ([t2_step5_c4_fix3_run.ipynb](t2_step5_c4_fix3_run.ipynb)) changes those requests and adds a last guard
  that replaces any number still flagged. **Its expectation, fixed before it ran:** 0 flagged numbers under rule 2,
  with the guard replacing at most 3.

All runs are compared in `results/t2_step7/README.md`.

**Limits:** numbers written as words aren't read; a number in a shown passage counts as sourced even where the
passage uses it for something else; 1 to 3 pass as ranks (hot spot 1, safe area 2).

**Outputs** in `results/t2_step7/<run>/`: `numbers.csv` (every number: report, section, kind, verdict, sentence),
`summary.csv`, `README.md`.

In [1]:
# ── Cell 1: The runs to check, and the guideline sections ─────────────────────
import json
import re
import sys
from pathlib import Path

import pandas as pd

T2 = Path(r"D:\CSE499AB_project\Essential_task_delegation\Offline_RAG_KR2")
RUNS = {"t2_step5_c4": T2 / "results" / "t2_step5_c4" / "t2_step5_c4_internet_off",              # run 1
        "t2_step5_c4_fix": T2 / "results" / "t2_step5_c4_fix" / "t2_step5_c4_fix_internet_off",    # run 2: fixes (a), (b)
        "t2_step5_c4_fix2": T2 / "results" / "t2_step5_c4_fix2" / "t2_step5_c4_fix2_internet_off",  # run 3: rule 2 too
        "t2_step5_c4_fix3": T2 / "results" / "t2_step5_c4_fix3" / "t2_step5_c4_fix3_internet_off"}  # run 4: requests, guard
RUNS = {name: run for name, run in RUNS.items() if run.is_dir()}       # a run not downloaded yet is left out
RULES = (1, 2)                                                          # number_check's rule versions
OUT_ROOT = T2 / "results" / "t2_step7"
sys.path.insert(0, str(T2 / "disasteller_offline"))
import number_check as nc

GRADES_HEADING = "Disaster grade of each location"      # fixed_flow.GRADES_HEADING: written by the code, not checked
# From fixed_flow.REPORTS: which reports also get the code's map reading, and the EMS-98 sections. The Expert team's
# report finds those with its own five searches; the reconstruction plan reuses them ("ems98": True).
SPECS = {"Expert_team_report.txt": {"map": False, "ems98": True},
         "Alerts_team_report_for_social_media.txt": {"map": True, "ems98": False},
         "Emergency_team_report.txt": {"map": True, "ems98": False},
         "Emergency_service_human_resource_allocation.txt": {"map": False, "ems98": False},
         "Public_community_report.txt": {"map": False, "ems98": False},
         "Reconstruction_plan.txt": {"map": False, "ems98": True}}
CHUNKS = {c["id"]: c for c in map(json.loads, open(T2 / "guideline_chunks.jsonl", encoding="utf-8"))}


def passage(section_id):
    """One search result, as the search tool printed it (guideline_index.format_results)."""
    c = CHUNKS[section_id]
    pages = f"p. {c['page_start']}" if c["page_start"] == c["page_end"] else f"pp. {c['page_start']}-{c['page_end']}"
    return f"[{section_id}] ({c['doc']}, {pages})\n{c['text']}"


for name, run in RUNS.items():
    scenes = sorted(p.name for p in run.glob("run_*"))
    print(f"✓ {name}: {len(scenes)} scenes, {sum(len(list((run / s / 'reports' / 'offline').glob('*.txt'))) for s in scenes)} reports")
print(f"✓ {len(CHUNKS)} guideline sections; number_check's rules from {nc.__file__}")

✓ t2_step5_c4: 2 scenes, 12 reports
✓ t2_step5_c4_fix: 2 scenes, 12 reports
✓ t2_step5_c4_fix2: 2 scenes, 12 reports
✓ t2_step5_c4_fix3: 2 scenes, 12 reports
✓ 405 guideline sections; number_check's rules from D:\CSE499AB_project\Essential_task_delegation\Offline_RAG_KR2\disasteller_offline\number_check.py


In [2]:
# ── Cell 2: Each report's sources, rebuilt from the run's own logs ────────────────
def between(text, start, end="\n\nReport: "):
    i = text.index(start) + len(start)
    return text[i:text.index(end, i)].strip()


def report_sources(scene_dir):
    """{report: the facts text it was given, without the photo descriptions and the earlier reports}."""
    log = (scene_dir / "run_log.txt").read_text(encoding="utf-8")
    facts = between(log, "C4's damage facts, given to every report:\n")
    reading = between(log, "Map reading made by the code from the hot spots:\n")
    searched, pending = {}, []
    for rec in map(json.loads, open(scene_dir / "llm_calls.jsonl", encoding="utf-8")):
        if rec["kind"] == "search":
            pending += rec["ids"]
        elif rec.get("report") and pending:
            searched.setdefault(rec["report"], []).extend(pending)
            pending = []
    ems98 = list(dict.fromkeys(searched.get("Expert_team_report.txt", [])))
    out = {}
    for report, spec in SPECS.items():
        ids = list(dict.fromkeys(searched.get(report, []) + (ems98 if spec["ems98"] else [])))
        parts = [("damage facts for the photo pair (from the system's damage model; estimates)", facts)]
        if spec["map"]:
            parts.append(("map reading (made by the code from the damage model's hot spots and safe areas)", reading))
        if ids:
            parts.append(("guideline sections (from the guideline search; cite these IDs)", "\n\n".join(map(passage, ids))))
        out[report] = "\n\n".join(f"=== Facts: {t} ===\n{b}" for t, b in parts)
    return out, searched


for name, run in RUNS.items():
    for scene_dir in sorted(run.glob("run_*")):
        sources, searched = report_sources(scene_dir)
        print(f"✓ {name} / {scene_dir.name[4:]}: guideline sections shown per report: " +
              "; ".join(f"{r[:-4]} {len(set(searched.get(r, [])))}" for r in SPECS))
        fact_text, _ = nc.sources_of(sources["Expert_team_report.txt"])
        k = nc.known(fact_text, "")
        print(f"  counts {sorted(k['count'])}, positions {sorted(k['coord'])}, pixel figures {sorted(k['px'])}")

✓ t2_step5_c4 / santa_rosa_wildfire_00000155_post_disaster: guideline sections shown per report: Expert_team_report 6; Alerts_team_report_for_social_media 0; Emergency_team_report 9; Emergency_service_human_resource_allocation 12; Public_community_report 9; Reconstruction_plan 9
  counts [3, 45, 96, 144], positions [(100, 773), (100, 923), (250, 923), (991, 779)], pixel figures [251.0, 300.0, 386.0, 1024.0, 684835.0]
✓ t2_step5_c4 / turkey_earthquake4_post_327: guideline sections shown per report: Expert_team_report 6; Alerts_team_report_for_social_media 0; Emergency_team_report 9; Emergency_service_human_resource_allocation 12; Public_community_report 9; Reconstruction_plan 9
  counts [11, 32, 54, 97], positions [(48, 593), (320, 32), (722, 694), (821, 807), (917, 923), (991, 32)], pixel figures [254.0, 311.0, 412.0, 1024.0, 22107.0, 28381.0, 403997.0]
✓ t2_step5_c4_fix / santa_rosa_wildfire_00000155_post_disaster: guideline sections shown per report: Expert_team_report 6; Alerts_team

In [3]:
# ── Cell 3: Every number in every model-written section, under rule versions 1 and 2 ──
VERDICTS = ["ok", "wrong", "unsourced", "misused"]
rows = []
for name, run in RUNS.items():
    for scene_dir in sorted(run.glob("run_*")):
        sources, _ = report_sources(scene_dir)
        for report in SPECS:
            text = (scene_dir / "reports" / "offline" / report).read_text(encoding="utf-8")
            k = nc.known(*nc.sources_of(sources[report]))
            parts = re.split(r"^## (.+)$", text, flags=re.MULTILINE)
            for heading, body in zip(parts[1::2], parts[2::2]):
                if heading.strip() == GRADES_HEADING:
                    continue
                for rule in RULES:
                    for n in nc.check(body, k, version=rule):
                        rows.append({"run": name, "rule": rule, "scene": scene_dir.name[4:], "report": report,
                                     "section": heading.strip(), **n})
NUMBERS = pd.DataFrame(rows)
SUMMARY = (NUMBERS.groupby(["run", "rule", "scene", "report"])["verdict"].value_counts().unstack(fill_value=0)
           .reindex(columns=VERDICTS, fill_value=0).reset_index())
SUMMARY["numbers"] = SUMMARY[VERDICTS].sum(axis=1)
print(SUMMARY[SUMMARY["rule"] == 2].drop(columns="rule").to_string(index=False))
TOTALS = SUMMARY.groupby(["run", "rule", "scene"])[["numbers"] + VERDICTS].sum().reset_index()
print("\n" + TOTALS.to_string(index=False))

             run                                      scene                                          report  ok  wrong  unsourced  misused  numbers
     t2_step5_c4 santa_rosa_wildfire_00000155_post_disaster         Alerts_team_report_for_social_media.txt   4      2          0        0        6
     t2_step5_c4 santa_rosa_wildfire_00000155_post_disaster Emergency_service_human_resource_allocation.txt  17      2          5        0       24
     t2_step5_c4 santa_rosa_wildfire_00000155_post_disaster                       Emergency_team_report.txt   5      0          0        0        5
     t2_step5_c4 santa_rosa_wildfire_00000155_post_disaster                          Expert_team_report.txt   3      3          0        0        6
     t2_step5_c4 santa_rosa_wildfire_00000155_post_disaster                     Public_community_report.txt   5      3          0        0        8
     t2_step5_c4 santa_rosa_wildfire_00000155_post_disaster                         Reconstruction_plan.txt   9 

In [4]:
# ── Cell 4: The rule, every flagged number, and the files ─────────────────────────
pd.set_option("display.max_colwidth", 160)


def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


def verdict(name, rule):
    """Rule 1: no wrong number. Rule 2: no wrong and no misused number."""
    mine = NUMBERS[(NUMBERS["run"] == name) & (NUMBERS["rule"] == rule)]
    return "passes" if not mine["verdict"].isin(["wrong", "misused"]).any() else "fails"


for name in RUNS:
    out = OUT_ROOT / name
    out.mkdir(parents=True, exist_ok=True)
    mine = NUMBERS[NUMBERS["run"] == name]
    lines = []
    for rule in RULES:
        r = mine[mine["rule"] == rule]
        counts = r["verdict"].value_counts().reindex(VERDICTS, fill_value=0)
        line = (f"rule {rule}: {len(r)} numbers, {counts['wrong']} wrong, {counts['unsourced']} unsourced, "
                f"{counts['misused']} misused: {verdict(name, rule).upper()}")
        print(f"===== {name}, {line}")
        lines.append(f"- **{line[0].upper() + line[1:]}**")
    flagged = mine[(mine["rule"] == 2) & (mine["verdict"] != "ok")][["scene", "report", "section", "token", "kind",
                                                                        "verdict", "sentence"]]
    print(flagged.to_string(index=False) if len(flagged) else "(nothing flagged under rule 2)")
    mine.to_csv(out / "numbers.csv", index=False)
    SUMMARY[SUMMARY["run"] == name].to_csv(out / "summary.csv", index=False)
    (out / "README.md").write_text("\n".join([
        f"# T2 step 7: every number in `{name}`'s reports", "",
        f"Made by [t2_step7_number_check.ipynb](../../../t2_step7_number_check.ipynb) with "
        "[number_check.py](../../../disasteller_offline/number_check.py); how the sources are rebuilt and how each "
        "number is checked are in the notebook's first cell. Rule 1 passes a run with no wrong number; rule 2 "
        "(20 Nov, evening) with no wrong and no misused number (a damage count used for staff, teams or shelters).", "",
        *lines, "",
        "## By report, rule 2", "", md(SUMMARY[(SUMMARY["run"] == name) & (SUMMARY["rule"] == 2)].drop(columns=["run", "rule"])), "",
        "## Every number flagged under rule 2", "", md(flagged) if len(flagged) else "None.", "",
        "Files: `numbers.csv` (every number under both rules, with its verdict and sentence), `summary.csv`.",
    ]) + "\n", encoding="utf-8")
    print(f"✓ Wrote README.md, numbers.csv and summary.csv to {out}\n")

# The runs side by side, under both rules
side = TOTALS.copy()
side["verdict"] = [verdict(n, r) for n, r in zip(side["run"], side["rule"])]
print(side.to_string(index=False))
(OUT_ROOT / "README.md").write_text("\n".join([
    "# T2 step 7: the numbers in T2 step 5's runs, side by side", "",
    "Made by [t2_step7_number_check.ipynb](../../t2_step7_number_check.ipynb).",
    "- Run 1 (`t2_step5_c4`) is the first run.",
    "- Run 2 (`t2_step5_c4_fix`) adds fix (a), clearer facts, and fix (b), the number check while writing.",
    "- Run 3 (`t2_step5_c4_fix2`), if listed, adds rule 2 to the check while writing.",
    "- Run 4 (`t2_step5_c4_fix3`), if listed, adds requests that don't ask for numbers the facts lack, and a last guard.", "",
    "**Rule 1** (fixed before run 1) passes a run with no wrong number. **Rule 2** (20 Nov, evening, fixed before "
    "run 3) also fails it for a misused number: a damage count used for staff, teams, shelters or other resources.", "",
    md(side), "",
    "Every number, with its verdict under both rules and its sentence, is in each run's folder (`numbers.csv`, `README.md`).",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote the side-by-side README.md to {OUT_ROOT}")

===== t2_step5_c4, rule 1: 129 numbers, 13 wrong, 5 unsourced, 0 misused: FAILS
===== t2_step5_c4, rule 2: 129 numbers, 13 wrong, 5 unsourced, 0 misused: FAILS
                                     scene                                          report                                            section         token   kind   verdict                                                                                                                                                                                    sentence
santa_rosa_wildfire_00000155_post_disaster                          Expert_team_report.txt                                           Overview           96%  share     wrong                                             The wildfire caused extensive destruction, with 96% of the area classified as destroyed, indicating total or near-total collapse of structures.
santa_rosa_wildfire_00000155_post_disaster                          Expert_team_report.txt Description of the photos a

===== t2_step5_c4_fix2, rule 1: 192 numbers, 0 wrong, 1 unsourced, 0 misused: PASSES
===== t2_step5_c4_fix2, rule 2: 192 numbers, 0 wrong, 1 unsourced, 9 misused: FAILS
                      scene                                          report                     section token  kind   verdict                                                                                                                                                                                                                                            sentence
turkey_earthquake4_post_327                       Emergency_team_report.txt          Emergency shelters    54 other   misused                                                                                                               The number of emergency shelters needed is 54, each accommodating one person, ensuring adequate living space for displaced residents.
turkey_earthquake4_post_327 Emergency_service_human_resource_allocation.txt Human Resource 

✓ Wrote README.md, numbers.csv and summary.csv to D:\CSE499AB_project\Essential_task_delegation\Offline_RAG_KR2\results\t2_step7\t2_step5_c4_fix2

===== t2_step5_c4_fix3, rule 1: 178 numbers, 0 wrong, 0 unsourced, 0 misused: PASSES
===== t2_step5_c4_fix3, rule 2: 178 numbers, 0 wrong, 0 unsourced, 0 misused: PASSES
(nothing flagged under rule 2)
✓ Wrote README.md, numbers.csv and summary.csv to D:\CSE499AB_project\Essential_task_delegation\Offline_RAG_KR2\results\t2_step7\t2_step5_c4_fix3

             run  rule                                      scene  numbers  ok  wrong  unsourced  misused verdict
     t2_step5_c4     1 santa_rosa_wildfire_00000155_post_disaster       60  43     12          5        0   fails
     t2_step5_c4     1                turkey_earthquake4_post_327       69  68      1          0        0   fails
     t2_step5_c4     2 santa_rosa_wildfire_00000155_post_disaster       60  43     12          5        0   fails
     t2_step5_c4     2                turkey_eart